# ERP Planner Workspace & Replenishment Decision Support

This notebook evaluates inventory-planning and replenishment workflows using ERP-derived operational datasets.

The objective is to support planner decision-making through inventory visibility, exception management, replenishment monitoring, supplier review, and inventory-risk assessment.

The analysis focuses on:

- inventory exceptions
- replenishment requirements
- planner review queues
- supplier exposure
- inventory-risk escalation
- planner-support KPIs

Rather than simulating ERP transactions, the notebook analyses ERP-style operational datasets and develops decision-support outputs commonly used in inventory-planning and materials-management activities.

In [17]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

POWERBI_INPUT_DIR = Path("../outputs/powerbi_operational")
OUTPUT_DIR = Path("../outputs/erp_reporting")

OUTPUT_DIR.mkdir(exist_ok=True)

## Load ERP Operational Datasets

The analysis uses inventory, replenishment, supplier, and material datasets prepared throughout the repository.

These datasets represent ERP-derived operational information commonly used by inventory planners and supply-chain analysts.

In [18]:
fact_inventory_status = pd.read_csv(
    POWERBI_INPUT_DIR / "fact_inventory_status.csv"
)

fact_purchase_requisitions = pd.read_csv(
    POWERBI_INPUT_DIR / "fact_purchase_requisitions.csv"
)

dim_parts = pd.read_csv(
    POWERBI_INPUT_DIR / "dim_parts.csv"
)

dim_suppliers = pd.read_csv(
    POWERBI_INPUT_DIR / "dim_suppliers.csv"
)

## Inventory Planner Workspace

Inventory planners require visibility of inventory availability, replenishment requirements, inventory criticality, and operational importance.

This section consolidates inventory and material information into a planner-oriented operational view.

In [19]:
planner_workspace = (
    fact_inventory_status
    .merge(
        dim_parts[
            [
                "Part_ID",
                "Part_Name",
            ]
        ],
        on="Part_ID",
        how="left"
    )
)

planner_workspace.head()

,Part_ID,On_Hand_Qty,On_Order_Qty,Allocated_Qty,Available_Qty,Avg_Weekly_Demand,Demand_SD,Demand_CV,Nonzero_Demand_Weeks,XYZ_Class,Lead_Time_Weeks,Safety_Stock_Qty,Reorder_Point_Qty,Suggested_Replenishment_Qty,Planner_Exception_Flag,Manual_Location,UDC_Type,Unit_Cost_EUR,ABC_Class,Movement_Class,Movement_Lines_36M,Criticality,Supplier_ID,Supplier_Region,Part_Category,Unit_Volume_cm3,Stock_Qty,Max_Stock_Qty,Stock_Managed,Estimated_Inventory_Value_EUR,Below_Reorder_Point,Stock_Coverage_Weeks,Total_Occupied_Volume_cm3,Part_Name
0,TC-CON-0343,46,0,9,37,3.000000,3.270592,1.090197,102,Y,1.4,18,35,0,False,R-I38,VER,40.87,A,A+,473,Medium,SUP-015,EU,Consumables & service kits,72293,46,48,1,1512.19,False,12.333333,3325478,Basic Line TC Grease Cartridge
1,VL-CON-0336,23,0,7,16,3.852564,2.248453,0.583625,139,X,1.0,13,22,22,True,R-B17,D,29.16,A,A+,395,Medium,SUP-004,EU,Consumables & service kits,470,23,38,1,466.56,True,4.153078,10810,Scissor Lift Air Filter
2,TC-CON-0329,23,0,5,18,3.365385,2.032277,0.603877,135,X,1.3,13,22,21,True,R-E25,B,43.54,A,A+,313,Medium,SUP-018,EU,Consumables & service kits,77,23,39,1,783.72,True,5.348571,1771,A2024 LL Air Filter
3,WA-CON-0631,16,7,4,19,3.602564,2.118062,0.587932,141,X,1.3,15,21,13,True,R-F76,D,26.96,A,A+,231,Medium,SUP-020,EU,Consumables & service kits,2480,16,32,1,512.24,True,5.274021,39680,Exact Precision Lubrication Kit
4,TC-CON-0171,17,0,2,15,2.929487,3.428744,1.170425,95,Y,1.0,6,12,0,False,R-L86,B,40.68,A,A+,227,Medium,SUP-003,EU,Consumables & service kits,820,17,20,1,610.20,False,5.120350,13940,Basic Line TC Air Filter


## Inventory Exception Monitoring

Inventory exceptions represent situations requiring planner review and potential replenishment actions.

Typical examples include:

- stock below reorder point
- safety-stock violations
- inventory shortages
- replenishment requirements

In [20]:
planner_workspace["Below_Reorder_Point"] = (
    planner_workspace["Available_Qty"]
    < planner_workspace["Reorder_Point_Qty"]
)

planner_workspace["Below_Safety_Stock"] = (
    planner_workspace["Available_Qty"]
    < planner_workspace["Safety_Stock_Qty"]
)

planner_workspace["Planner_Exception"] = (
    planner_workspace["Below_Reorder_Point"]
    |
    planner_workspace["Below_Safety_Stock"]
)

exception_summary = pd.DataFrame({
    "Metric": [
        "Parts Below Reorder Point",
        "Parts Below Safety Stock",
        "Planner Exceptions"
    ],
    "Value": [
        planner_workspace["Below_Reorder_Point"].sum(),
        planner_workspace["Below_Safety_Stock"].sum(),
        planner_workspace["Planner_Exception"].sum()
    ]
})

exception_summary

,Metric,Value
0,Parts Below Reorder Point,47
1,Parts Below Safety Stock,2
2,Planner Exceptions,47


## Planner Prioritisation Queue

Not all inventory exceptions require the same level of attention.

This section creates a prioritised planner review queue using inventory criticality, movement intensity, and inventory classification.

In [21]:
criticality_score = {
    "Low": 1,
    "Medium": 2,
    "High": 3,
    "Critical": 4
}

movement_score = {
    "NC": 0,
    "D": 1,
    "C": 2,
    "B": 3,
    "A": 4,
    "A+": 5
}

abc_score = {
    "C": 1,
    "B": 2,
    "A": 3
}

planner_exception_queue = (
    planner_workspace[
        planner_workspace["Planner_Exception"]
    ]
    .copy()
)

planner_exception_queue["Priority_Score"] = (
    planner_exception_queue["Criticality"].map(criticality_score)
    +
    planner_exception_queue["Movement_Class"].map(movement_score)
    +
    planner_exception_queue["ABC_Class"].map(abc_score)
)

planner_exception_queue["Suggested_Planner_Action"] = (
    planner_exception_queue.apply(
        planner_action,
        axis=1
    )
)

planner_exception_queue = (
    planner_exception_queue
    .sort_values(
        "Priority_Score",
        ascending=False
    )
)

planner_exception_queue.head(15)

,Part_ID,On_Hand_Qty,On_Order_Qty,Allocated_Qty,Available_Qty,Avg_Weekly_Demand,Demand_SD,Demand_CV,Nonzero_Demand_Weeks,XYZ_Class,Lead_Time_Weeks,Safety_Stock_Qty,Reorder_Point_Qty,Suggested_Replenishment_Qty,Planner_Exception_Flag,Manual_Location,UDC_Type,Unit_Cost_EUR,ABC_Class,Movement_Class,Movement_Lines_36M,Criticality,Supplier_ID,Supplier_Region,Part_Category,Unit_Volume_cm3,Stock_Qty,Max_Stock_Qty,Stock_Managed,Estimated_Inventory_Value_EUR,Below_Reorder_Point,Stock_Coverage_Weeks,Total_Occupied_Volume_cm3,Part_Name,Below_Safety_Stock,Planner_Exception,Priority_Score,Suggested_Planner_Action
6,VL-CON-0419,12,8,2,18,2.397436,2.983134,1.244302,85,Y,1.3,13,19,8,True,R-H98,D,20.00,A,A+,196,High,SUP-020,EU,Consumables & service kits,2512,12,26,1,360.00,True,7.508021,30144,ERCO Lift O-Ring Kit,False,True,11,Create Requisition
1,VL-CON-0336,23,0,7,16,3.852564,2.248453,0.583625,139,X,1.0,13,22,22,True,R-B17,D,29.16,A,A+,395,Medium,SUP-004,EU,Consumables & service kits,470,23,38,1,466.56,True,4.153078,10810,Scissor Lift Air Filter,False,True,10,Create Requisition
3,WA-CON-0631,16,7,4,19,3.602564,2.118062,0.587932,141,X,1.3,15,21,13,True,R-F76,D,26.96,A,A+,231,Medium,SUP-020,EU,Consumables & service kits,2480,16,32,1,512.24,True,5.274021,39680,Exact Precision Lubrication Kit,False,True,10,Create Requisition
2,TC-CON-0329,23,0,5,18,3.365385,2.032277,0.603877,135,X,1.3,13,22,21,True,R-E25,B,43.54,A,A+,313,Medium,SUP-018,EU,Consumables & service kits,77,23,39,1,783.72,True,5.348571,1771,A2024 LL Air Filter,False,True,10,Create Requisition
19,AD-CON-0798,8,4,1,11,0.993590,1.517625,1.527416,67,Y,1.4,11,17,13,True,R-S24,B,30.37,A,B,55,High,SUP-005,Eastern Europe,Consumables & service kits,305,8,24,1,334.07,True,11.070968,2440,Radar Calibration Lubrication Kit,False,True,9,Create Requisition
36,WB-CON-0813,12,3,2,13,1.089744,1.782972,1.636139,55,Y,1.0,10,14,8,True,R-Y74,E,32.38,A,B,39,Medium,SUP-019,EU,Consumables & service kits,2764,12,21,1,420.94,True,11.929412,33168,Eyelight Rubber Pad,False,True,8,Create Requisition
38,TC-CON-0347,9,3,0,12,0.448718,1.266414,2.822294,23,Z,1.6,10,16,11,True,R-N62,B,18.94,A,B,37,Medium,SUP-014,Eastern Europe,Consumables & service kits,194,9,23,1,227.28,True,26.742857,1746,A2024 LL Protective Cover,False,True,8,Create Requisition
187,WB-CON-0955,3,3,0,6,0.493590,1.177733,2.386057,36,Y,1.0,4,7,5,True,R-M27,F,18.05,B,C,4,Critical,SUP-009,Italy,Consumables & service kits,33247,3,11,1,108.30,True,12.155844,99741,EM 9000 Rubber Pad,False,True,8,Create Requisition
172,TE-BRG-0020,4,2,0,6,0.192308,0.745874,3.878543,16,Z,1.3,5,7,4,True,R-N36,F,105.00,A,C,5,Medium,SUP-018,EU,Bearings & shafts,29296,4,10,1,630.00,True,31.200000,117184,Brake Tester Spindle Bearing,False,True,7,Create Requisition
32,VL-CON-0898,12,4,0,16,0.955128,1.779246,1.862834,52,Y,1.1,11,17,10,True,R-F98,B,3.27,C,B,44,High,SUP-008,EU,Consumables & service kits,367,12,26,1,52.32,True,16.751678,4404,ERCO Lift Protective Cover,False,True,7,Create Requisition


## Replenishment Review Queue

The purchase requisition dataset represents replenishment requirements identified through inventory planning activities and awaiting planner review.

This section evaluates the current replenishment workload.

In [22]:
requisition_summary = pd.DataFrame({
    "Metric": [
        "Open Requisitions",
        "Total Requisition Value EUR",
        "Average Requisition Value EUR"
    ],
    "Value": [
        len(fact_purchase_requisitions),
        round(
            fact_purchase_requisitions["Estimated_Value_EUR"].sum(),
            2
        ),
        round(
            fact_purchase_requisitions["Estimated_Value_EUR"].mean(),
            2
        )
    ]
})

requisition_summary

,Metric,Value
0,Open Requisitions,47.00
1,Total Requisition Value EUR,15003.97
2,Average Requisition Value EUR,319.23


## Requisition Priority Exposure

This section evaluates replenishment requirements according to operational priority and associated inventory value.

In [23]:
priority_summary = (
    fact_purchase_requisitions
    .groupby("Priority")
    .agg(
        Requisitions=("Purchase_Requisition_ID", "count"),
        Total_Value_EUR=("Estimated_Value_EUR", "sum")
    )
    .reset_index()
)

priority_summary["Value_Share_%"] = (
    priority_summary["Total_Value_EUR"]
    /
    priority_summary["Total_Value_EUR"].sum()
    * 100
).round(2)

priority_summary["Avg_Requisition_Value"] = (
    priority_summary["Total_Value_EUR"]
    /
    priority_summary["Requisitions"]
).round(2)

priority_summary

,Priority,Requisitions,Total_Value_EUR,Value_Share_%,Avg_Requisition_Value
0,High,7,5995.34,39.96,856.48
1,Low,27,7518.13,50.11,278.45
2,Medium,13,1490.50,9.93,114.65


### Operational Observation

The replenishment workload remains concentrated among a relatively small number of high-priority requisitions.

Although high-priority requisitions account for only a limited share of the total requisition population, they represent the highest average replenishment value per requisition and therefore require close planner attention.

Conversely, low-priority requisitions represent routine replenishment activity distributed across a larger number of materials and contribute most of the overall replenishment workload.

## Supplier Replenishment Exposure

Supplier performance directly influences inventory availability and replenishment responsiveness.

This section evaluates supplier exposure across the replenishment workload.

In [24]:
supplier_exposure = (
    fact_purchase_requisitions
    .merge(
        dim_suppliers,
        on="Supplier_ID",
        how="left"
    )
)

supplier_replenishment_exposure = (
    supplier_exposure
    .groupby(
        [
            "Supplier_ID",
            "Supplier_Name"
        ]
    )
    .agg(
        Requisitions=("Purchase_Requisition_ID", "count"),
        Requisition_Value_EUR=("Estimated_Value_EUR", "sum"),
        Avg_Lead_Time_Days=("Avg_Lead_Time_Days", "mean"),
        Reliability_Score=("Reliability_Score", "mean")
    )
    .round(2)
    .reset_index()
)

supplier_replenishment_exposure["Supplier_Risk"] = np.where(
    (
        supplier_replenishment_exposure["Reliability_Score"] < 0.85
    )
    &
    (
        supplier_replenishment_exposure["Avg_Lead_Time_Days"] > 8
    ),
    "Review",
    "Normal"
)

supplier_replenishment_exposure.sort_values(
    "Requisition_Value_EUR",
    ascending=False
).head(15)

,Supplier_ID,Supplier_Name,Requisitions,Requisition_Value_EUR,Avg_Lead_Time_Days,Reliability_Score,Supplier_Risk
14,SUP-016,EU Industrial Components 16,4,3802.14,8.0,0.86,Normal
18,SUP-020,EU Industrial Components 20,5,2510.48,9.0,0.98,Normal
16,SUP-018,EU Industrial Components 18,7,2397.43,9.0,0.83,Review
3,SUP-004,EU Industrial Components 04,5,1795.46,7.0,0.95,Normal
4,SUP-005,Eastern Europe Industrial Components 05,4,1050.00,10.0,0.75,Review
6,SUP-007,Eastern Europe Industrial Components 07,2,1048.12,10.0,0.91,Normal
11,SUP-013,North America Industrial Components 13,2,504.36,19.0,0.86,Normal
2,SUP-003,EU Industrial Components 03,2,318.36,6.0,0.85,Normal
9,SUP-010,EU Industrial Components 10,4,289.24,8.0,0.92,Normal
17,SUP-019,EU Industrial Components 19,2,283.68,4.0,0.89,Normal


## Planner Action Review

Inventory exceptions are translated into recommended planner actions according to inventory risk and replenishment urgency.

The majority of materials remain within acceptable inventory-control thresholds and require routine monitoring.

Only a limited number of materials require replenishment intervention, while a very small subset of inventory positions require expedited action due to safety-stock breaches.

This prioritised approach enables planners to focus attention on operationally relevant inventory risks while maintaining visibility across the broader material population.

In [25]:
def planner_action(row):

    if row["Available_Qty"] < row["Safety_Stock_Qty"]:
        return "Expedite Supply"

    if row["Available_Qty"] < row["Reorder_Point_Qty"]:
        return "Create Requisition"

    return "Monitor"


planner_workspace["Suggested_Planner_Action"] = (
    planner_workspace.apply(
        planner_action,
        axis=1
    )
)

planner_action_summary = (
    planner_workspace
    .groupby("Suggested_Planner_Action")
    .agg(
        Parts=("Part_ID", "count")
    )
    .reset_index()
)


planner_action_summary

,Suggested_Planner_Action,Parts
0,Create Requisition,45
1,Expedite Supply,2
2,Monitor,953


## Planner KPI Dashboard

The following KPIs summarise the operational workload associated with inventory planning and replenishment management.

In [26]:
planner_kpi_dashboard = pd.DataFrame({
    "Metric": [
        "Inventory Parts Reviewed",
        "Planner Exceptions",
        "Exception Rate %",
        "Open Requisitions",
        "Requisition Value EUR",
        "Expedite Actions",
        "High Priority Requisitions"
    ],
    "Value": [
        len(planner_workspace),
        planner_workspace["Planner_Exception"].sum(),
        round(
            planner_workspace["Planner_Exception"].mean() * 100,
            2
        ),
        len(fact_purchase_requisitions),
        round(
            fact_purchase_requisitions["Estimated_Value_EUR"].sum(),
            2
        ),
        (
            planner_workspace["Suggested_Planner_Action"]
            == "Expedite Supply"
        ).sum(),
        (
            fact_purchase_requisitions["Priority"]
            == "High"
        ).sum()
    ]
})

planner_kpi_dashboard

,Metric,Value
0,Inventory Parts Reviewed,1000.00
1,Planner Exceptions,47.00
2,Exception Rate %,4.70
3,Open Requisitions,47.00
4,Requisition Value EUR,15003.97
5,Expedite Actions,2.00
6,High Priority Requisitions,7.00


## Export ERP Reporting Tables

The outputs generated in this notebook provide Power BI-ready reporting tables supporting planner dashboards, replenishment monitoring, supplier review, and inventory-risk reporting.

In [27]:
planner_exception_queue.to_csv(
    OUTPUT_DIR / "planner_exception_queue.csv",
    index=False
)

priority_summary.to_csv(
    OUTPUT_DIR / "requisition_priority_summary.csv",
    index=False
)

supplier_replenishment_exposure.to_csv(
    OUTPUT_DIR / "supplier_replenishment_exposure.csv",
    index=False
)

planner_action_summary.to_csv(
    OUTPUT_DIR / "planner_action_summary.csv",
    index=False
)

planner_kpi_dashboard.to_csv(
    OUTPUT_DIR / "planner_kpi_dashboard.csv",
    index=False
)

## Operational Observation

Most materials remain within acceptable inventory-control thresholds and require only routine monitoring.

A limited number of inventory exceptions generate the majority of planner workload, while only a very small subset of materials require urgent intervention due to safety-stock breaches.

This concentration of replenishment activity supports a prioritised inventory-management approach focused on operationally relevant spare parts and material-availability risks.